# Ejercicio: Web Scraping

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [1]:
from bs4 import BeautifulSoup

file = '/content/sample_data/rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()

# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [2]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [3]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [4]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [5]:
# Find all the links to other recipes
recipe_links = soup.find_all("a", href=True)

# Filter and print only the links that are likely to be recipes
recipe_urls = []
for link in recipe_links:
    href = link['href']
    if "recipe" in href:
        recipe_urls.append(href)

# Print the recipe URLs
print("Linked Recipes:")
for url in recipe_urls:
    print(url)

Linked Recipes:
https://www.allrecipes.com/authentication/login?regSource=3675&relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
/account/add-recipe
https://www.myrecipes.com/favorites
https://www.allrecipes.com/authentication/logout?relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
https://www.magazines.com/allrecipes-magazine.html?utm_source=allrecipes.com&utm_medium=owned&utm_campaign=i111arr1w2661
https://www.magazines.com/allrecipes-magazine.html
https://www.allrecipes.com/recipes/17562/dinner/
https://www.allrecipes.com/recipes/17057/everyday-cooking/more-meal-ideas/5-ingredients/main-dishes/
https://www.allrecipes.com/recipes/15436/everyday-cooking/one-pot-meals/
https://www.allrecipes.com/recipes/1947/everyday-cooking/quick-and-easy/
https://www.allrecipes.com/recipes/455/everyday-cooking/more-meal-ideas/30-minute-meals/
https://www.allrecipes.com/recipes/17889/everyday-cooking/family-friendly/family-dinners/
https://www.allrecipes.com/recipes/94/soups-s

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [6]:
%%capture
!pip install -q langchain langchain-community sentence-transformers faiss-cpu google-generativeai

### 4.1 Preparar el Corpus y Embeddings
Convertiremos la información extraída (título, descripción, ingredientes e instrucciones) en documentos procesables.

In [9]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings

# Unificamos la información en un solo texto o lista de documentos
corpus_text = f"Receta: {title}\nDescripción: {description}\n\nIngredientes:\n" + "\n".join(ingredients) + "\n\nInstrucciones:\n" + "\n".join(instructions)

# Dividimos por secciones para una mejor recuperación
documents = [
    Document(page_content=f"Título y Descripción: {title}. {description}", metadata={"source": "info"}),
    Document(page_content=f"Ingredientes: {', '.join(ingredients)}", metadata={"source": "ingredients"}),
    Document(page_content=f"Instrucciones de preparación: {' '.join(instructions)}", metadata={"source": "instructions"})
]

# Inicializamos los embeddings (usando un modelo ligero de HuggingFace)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Creamos la base de datos vectorial (Vector Store)
vector_db = FAISS.from_documents(documents, embeddings)

print("Base de datos vectorial creada con éxito.")

/tmp/ipykernel_585/1750565235.py:16: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Base de datos vectorial creada con éxito.


### 4.2 Configurar el Modelo de Lenguaje (Gemini)
Configurar la `GOOGLE_API_KEY` en los secretos de Colab.

In [25]:
import google.generativeai as genai
from google.colab import userdata

try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)

    # Listar modelos disponibles para depuración si es necesario
    # for m in genai.list_models():
    #     if 'generateContent' in m.supported_generation_methods:
    #         print(m.name)

    # gemini-1.5-flash fue descontinuado por Google; usamos 2.5-flash
    model_name = 'gemini-2.5-flash'
    model = genai.GenerativeModel(model_name)
    print(f"Modelo {model_name} configurado correctamente.")
except Exception as e:
    print(f"Error: {e}. Asegúrate de tener configurada la GOOGLE_API_KEY.")

Modelo gemini-2.5-flash configurado correctamente.


### 4.3 Función de Consulta RAG
Esta función busca la información relevante en el corpus y genera una respuesta usando Gemini.

In [26]:
def ask_recipe(question):
    # 1. Búsqueda de similitud
    docs = vector_db.similarity_search(question, k=2)
    context = "\n".join([d.page_content for d in docs])

    # 2. Construcción del Prompt
    prompt = f"""
    Eres un asistente de cocina. Utiliza el siguiente contexto de una receta para responder la pregunta del usuario.
    Si la información no está en el contexto, di que no lo sabes.

    Contexto:
    {context}

    Pregunta: {question}
    """

    # 3. Generación
    response = model.generate_content(prompt)
    return response.text

# Ejemplo de uso:
pregunta = "¿A qué temperatura debe estar el pollo internamente para estar listo?"
print(f"Pregunta: {pregunta}")
try:
    print(f"Respuesta: {ask_recipe(pregunta)}")
except Exception as e:
    print(f"Error al generar respuesta: {e}")

Pregunta: ¿A qué temperatura debe estar el pollo internamente para estar listo?
Respuesta: El pollo debe alcanzar una temperatura interna de al menos 180 grados F (83 grados C) para estar listo.
